## load the model

In [13]:
import torch
from assignment1.model import UNet

ckpt = torch.load("C:\\Users\\hanqingx\\projects\\diffusion-models\\checkpoints\\ddpm_epoch_49.pth", map_location="cpu")

print(type(ckpt))
if isinstance(ckpt, dict):
    print(ckpt.keys())

<class 'dict'>
dict_keys(['model', 'ema', 'optimizer', 'epoch'])


In [14]:
import torch
from assignment1.model import UNet

device = "cuda" if torch.cuda.is_available() else "cpu"

model = UNet().to(device)

ckpt = torch.load(
    "C:/Users/hanqingx/projects/diffusion-models/checkpoints/ddpm_epoch_49.pth",
    map_location=device
)

model.load_state_dict(ckpt["ema"])
model.eval()

UNet(
  (time_mlp): TimeEmbedding(
    (mlp): Sequential(
      (0): Linear(in_features=64, out_features=256, bias=True)
      (1): SiLU()
      (2): Linear(in_features=256, out_features=64, bias=True)
    )
  )
  (init_conv): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (down1): ModuleList(
    (0): ResidualBlock(
      (time_mlp): Sequential(
        (0): SiLU()
        (1): Linear(in_features=64, out_features=32, bias=True)
      )
      (conv1): Sequential(
        (0): GroupNorm(8, 32, eps=1e-05, affine=True)
        (1): SiLU()
        (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      )
      (conv2): Sequential(
        (0): GroupNorm(8, 32, eps=1e-05, affine=True)
        (1): SiLU()
        (2): Dropout(p=0.1, inplace=False)
        (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      )
      (residual_conv): Identity()
    )
    (1): Conv2d(32, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  )
 

## modify the sample function

In [4]:
T = 300

beta = torch.linspace(1e-4, 0.02, T).to(device)
alpha = 1.0 - beta
alpha_cumprod = torch.cumprod(alpha, dim=0)

In [5]:
import torch
from tqdm import tqdm

@torch.no_grad()
def sample_with_intermediates(
    model, n_samples, image_channels, image_size, T,
    alpha, alpha_cumprod, beta, device,
    save_steps=(300, 250, 200, 150, 100, 50, 0),
):
    model.eval()
    save_steps = set(save_steps)

    # Start from pure noise x_T
    x = torch.randn((n_samples, image_channels, image_size, image_size), device=device)

    x0_snapshots = {}  # {timestep: x0_hat_in_[0,1]}

    for i in tqdm(reversed(range(T)), total=T, desc="Sampling", leave=False):
        t = torch.full((n_samples,), i, device=device, dtype=torch.long)

        # Predict noise epsilon_theta(x_t, t)
        predicted_noise = model(x, t)

        # Scalars for this timestep (make sure they broadcast)
        alpha_t = alpha[i]
        alpha_cumprod_t = alpha_cumprod[i]
        beta_t = beta[i]

        # ---- compute x0_hat (model's prediction of final clean image) ----
        # x0_hat = (x_t - sqrt(1-a_bar_t)*eps) / sqrt(a_bar_t)
        x0_hat = (x - torch.sqrt(1.0 - alpha_cumprod_t) * predicted_noise) / torch.sqrt(alpha_cumprod_t)

        if i in save_steps:
            # store a visualization-friendly version in [0,1]
            x0_vis = (x0_hat.clamp(-1, 1) + 1) / 2
            x0_snapshots[i] = x0_vis.detach().cpu()  # move to CPU to save RAM/VRAM

        # ---- DDPM reverse step to get x_{t-1} ----
        if i > 0:
            noise = torch.randn_like(x)
        else:
            noise = 0

        mean = (1 / torch.sqrt(alpha_t)) * (
            x - ((1 - alpha_t) / torch.sqrt(1 - alpha_cumprod_t)) * predicted_noise
        )
        std = torch.sqrt(beta_t)
        x = mean + std * noise

    model.train()

    x_final = (x.clamp(-1, 1) + 1) / 2
    return x_final, x0_snapshots

In [10]:
x_final, x0_snaps = sample_with_intermediates(model, n_samples=5, image_channels=1, image_size=28, T=T, alpha=alpha, alpha_cumprod=alpha_cumprod, beta=beta, device=device)

## transfer intermediate states into images

In [8]:
from torchvision.utils import save_image, make_grid
import os

def save_snapshots(x0_snapshots, outdir="intermediates"):
    os.makedirs(outdir, exist_ok=True)
    for t, imgs in sorted(x0_snapshots.items(), reverse=True):
        # imgs: [N,C,H,W] in [0,1]
        grid = make_grid(imgs, nrow=min(5, imgs.shape[0]))
        save_image(grid, os.path.join(outdir, f"x0hat_t{t:03d}.png"))

In [11]:
save_snapshots(x0_snaps, outdir="x0hat_snaps")